In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from utils.utils import corrected_eventwise_F0_5_score


In [2]:
# Load channels 41-46 and ground truth from the training set
channel_names = [f"channel_{x}" for x in range(41,46 + 1)]
columns_to_load =  channel_names + ["is_anomaly"]
df = pd.read_parquet("data/train.parquet", columns=columns_to_load)
df = df.head(50000)


In [3]:
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idxs, val_idxs = next(splitter.split(df, df['is_anomaly']))


In [4]:
import torch
from torch.utils.data import Dataset

class FrameDataset(Dataset):
    def __init__(self, df, idxs, channel_names, label_name):
        self.idxs=idxs
        self.channel_names = channel_names
        self.label_name = label_name
        self.df = df

    def __len__(self):
        return len(self.idxs)

    def __getitem__(self, idx):
        X = torch.tensor(self.df.loc[self.idxs[idx], self.channel_names].values, dtype=torch.float32)
        y = torch.tensor(self.df.loc[self.idxs[idx], self.label_name], dtype=torch.long)
        return X, y


train_ds=FrameDataset(df, train_idxs, channel_names, 'is_anomaly')
val_ds=FrameDataset(df, val_idxs, channel_names, 'is_anomaly')


In [5]:
from fastai.data.core import DataLoader, DataLoaders

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
valid_dl = DataLoader(val_ds, batch_size=64)

dls = DataLoaders(train_dl, valid_dl)

In [17]:
import torch.nn as nn

class model(nn.Module):
    def __init__(self, n_inputs):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_inputs, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # 2 clases para clasificación binaria
        )

    def forward(self, x):  # `fastai` pasa así los inputs tabulares
        return self.net(x)

In [18]:
from fastai.tabular.all import tabular_learner
from fastai.basics import Learner
from fastai.metrics import accuracy
from torch.nn import CrossEntropyLoss
from torch.nn import BCEWithLogitsLoss

n_inputs = len(channel_names)
modelo_custom = model(n_inputs)
learn = Learner(dls, modelo_custom, loss_func=CrossEntropyLoss(), metrics=accuracy)

In [19]:
learn.fit_one_cycle(2)

epoch,train_loss,valid_loss,accuracy,time


KeyboardInterrupt: 